**Set environment**

In [1]:
suppressMessages(suppressWarnings(source("../run_config_project_sing.R")))
show_env()

You are working on        Singularity: singularity_proj_encode_fcc 
BASE DIRECTORY (FD_BASE): /data/reddylab/Kuei 
REPO DIRECTORY (FD_REPO): /data/reddylab/Kuei/repo 
WORK DIRECTORY (FD_WORK): /data/reddylab/Kuei/work 
DATA DIRECTORY (FD_DATA): /data/reddylab/Kuei/data 

You are working with      ENCODE FCC 
PATH OF PROJECT (FD_PRJ): /data/reddylab/Kuei/repo/Proj_ENCODE_FCC 
PROJECT RESULTS (FD_RES): /data/reddylab/Kuei/repo/Proj_ENCODE_FCC/results 
PROJECT SCRIPTS (FD_EXE): /data/reddylab/Kuei/repo/Proj_ENCODE_FCC/scripts 
PROJECT DATA    (FD_DAT): /data/reddylab/Kuei/repo/Proj_ENCODE_FCC/data 
PROJECT NOTE    (FD_NBK): /data/reddylab/Kuei/repo/Proj_ENCODE_FCC/notebooks 
PROJECT DOCS    (FD_DOC): /data/reddylab/Kuei/repo/Proj_ENCODE_FCC/docs 
PROJECT LOG     (FD_LOG): /data/reddylab/Kuei/repo/Proj_ENCODE_FCC/log 
PROJECT REF     (FD_REF): /data/reddylab/Kuei/repo/Proj_ENCODE_FCC/references 



**Set global variable**

In [2]:
vec = c(
    "fcc_astarr_macs_input_overlap",
    "fcc_astarr_macs_input_union"
)
names(vec) = vec

VEC_TXT_FOLDER = vec
for(txt in vec){cat(txt, "\n")}

fcc_astarr_macs_input_overlap 
fcc_astarr_macs_input_union 


**View files**

In [3]:
txt_folder = VEC_TXT_FOLDER[1]
txt_fdiry  = file.path(FD_RES, "analysis_fcc_comparison", txt_folder)
txt_fname  = "region.annotation.fcc_starrmpra_group.coverage.crispri.tsv"
txt_fglob  = file.path(txt_fdiry, txt_fname)

vec = Sys.glob(txt_fglob)
for (txt in vec) {cat(txt, "\n")}

/data/reddylab/Kuei/repo/Proj_ENCODE_FCC/results/analysis_fcc_comparison/fcc_astarr_macs_input_overlap/region.annotation.fcc_starrmpra_group.coverage.crispri.tsv 


In [4]:
txt_folder = VEC_TXT_FOLDER[1]
txt_fdiry = file.path(FD_RES, "region_closest", txt_folder, "summary")
txt_fname  = "region.annotation.genome_tss*tss_essential.tsv"
txt_fglob  = file.path(txt_fdiry, txt_fname)

vec = Sys.glob(txt_fglob)
for (txt in vec) {cat(txt, "\n")}

/data/reddylab/Kuei/repo/Proj_ENCODE_FCC/results/region_closest/fcc_astarr_macs_input_overlap/summary/region.annotation.genome_tss_pol2_rnaseq.tss_essential.tsv 
/data/reddylab/Kuei/repo/Proj_ENCODE_FCC/results/region_closest/fcc_astarr_macs_input_overlap/summary/region.annotation.genome_tss_pol2.tss_essential.tsv 


## Import data

### Import STARR/MPRA CREs screened by CRISPRi

In [5]:
### loop to import data
lst = lapply(VEC_TXT_FOLDER, function(txt_folder){
    ### set file directory
    txt_fdiry  = file.path(FD_RES, "analysis_fcc_comparison", txt_folder)
    txt_fname  = "region.annotation.fcc_starrmpra_group.coverage.crispri.tsv"
    txt_fpath = file.path(txt_fdiry, txt_fname)

    ### read table
    dat = read_tsv(txt_fpath, show_col_types = FALSE)
    return(dat)
})

### assign and show
lst_dat_region_merge_crispri_import = lst

res = lapply(lst, dim)
print(res)

dat = lst[[1]]
head(dat, 3)

$fcc_astarr_macs_input_overlap
[1] 15460    13

$fcc_astarr_macs_input_union
[1] 15847    13



Chrom,ChromStart,ChromEnd,Region,Group,Label,Score,Zcore,Score_Label,Zcore_Label,Assay_Folder,Assay_Group,Assay_Label
<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>
chr11,4092109,4092511,chr11:4092109-4092511,Distal:Active,Screen:NotSignif,-0.12943640,-0.12943640,Mean(ZScore),Mean(ZScore),CRISPRi_FlowFISH_K562_Riley_JinWoo,CRISPRi-HCRFF,CRISPRi-HCRFF
chr11,4094223,4095304,chr11:4094223-4095304,Proximal:Active,Screen:NotSignif,-0.04849565,-0.04849565,Mean(ZScore),Mean(ZScore),CRISPRi_FlowFISH_K562_Riley_JinWoo,CRISPRi-HCRFF,CRISPRi-HCRFF
chr11,4393045,4394087,chr11:4393045-4394087,Proximal:Active,Screen:NotSignif,-0.08214896,-0.08214896,Mean(ZScore),Mean(ZScore),CRISPRi_FlowFISH_K562_Riley_JinWoo,CRISPRi-HCRFF,CRISPRi-HCRFF


**Check: assay and label count**

In [6]:
lst = lst_dat_region_merge_crispri_import
lst = lapply(lst, function(dat){
    res = table(dat$Assay_Label, dat$Label, dnn=c("Assay", "Label"))
    dat = as.data.frame(res)
    return(dat)
})

dat = bind_rows(lst, .id = "Region")
dat = dat %>% tidyr::spread(Region, Freq)
fun_display_table(dat)

Assay,Label,fcc_astarr_macs_input_overlap,fcc_astarr_macs_input_union
CRISPRi-Growth,Screen:NotSignif,13761,14094
CRISPRi-Growth,Screen:Signif,1479,1517
CRISPRi-HCRFF,Screen:NotSignif,190,207
CRISPRi-HCRFF,Screen:Signif,30,29


### Import TSS Proximity and Essential

In [7]:
### loop to import data
lst = lapply(VEC_TXT_FOLDER, function(txt_folder){
    ### set file directory
    txt_fdiry = file.path(FD_RES, "region_closest", txt_folder, "summary")
    txt_fname = "region.annotation.genome_tss_pol2.tss_essential.tsv"
    txt_fpath = file.path(txt_fdiry, txt_fname)

    ### read table
    dat = read_tsv(txt_fpath, show_col_types = FALSE)
    return(dat)
})

### assign and show
lst_dat_region_annot_tss_import = lst

res = lapply(lst, dim)
print(res)

dat = lst[[1]]
fun_display_table(head(dat, 3))

$fcc_astarr_macs_input_overlap
[1] 150042      9

$fcc_astarr_macs_input_union
[1] 246852      9



Chrom,ChromStart,ChromEnd,Region,Annotation_A,Annotation_B,TSS_Proximity,TSS_Essential_Count,TSS_Category
chr1,10038,10405,chr1:10038-10405,fcc_astarr_macs_input_overlap,genome_tss_pol2,Proximal,0,Proximal:Non-Essential
chr1,14282,14614,chr1:14282-14614,fcc_astarr_macs_input_overlap,genome_tss_pol2,Distal,0,Distal
chr1,16025,16338,chr1:16025-16338,fcc_astarr_macs_input_overlap,genome_tss_pol2,Proximal,0,Proximal:Non-Essential


## Arrange table

### Get CRISPRi-Growth significant regions

In [8]:
lst = lst_dat_region_merge_crispri_import
lst = lapply(lst, function(dat){
    dat = dat %>% 
        dplyr::filter(Assay_Label == "CRISPRi-Growth") %>%
        dplyr::filter(Label == "Screen:Signif") %>%
        dplyr::filter(str_detect(Group, "Active"))
    return(dat)
})

### assign and show
lst_dat_region_merge_crispri_arrange = lst

res = lapply(lst, dim)
print(res)

dat = lst[[1]]
fun_display_table(head(dat, 3))

$fcc_astarr_macs_input_overlap
[1] 1428   13

$fcc_astarr_macs_input_union
[1] 1455   13



Chrom,ChromStart,ChromEnd,Region,Group,Label,Score,Zcore,Score_Label,Zcore_Label,Assay_Folder,Assay_Group,Assay_Label
chr1,1068587,1070616,chr1:1068587-1070616,Proximal:Active,Screen:Signif,0.2520571,0.2520571,Mean(ZScore),Mean(ZScore),CRISPRi_Growth_K562_Gersbach_JinWoo,CRISPRi-Growth,CRISPRi-Growth
chr1,1324275,1325765,chr1:1324275-1325765,Proximal:Active,Screen:Signif,0.5069515,0.5069515,Mean(ZScore),Mean(ZScore),CRISPRi_Growth_K562_Gersbach_JinWoo,CRISPRi-Growth,CRISPRi-Growth
chr1,1573998,1575422,chr1:1573998-1575422,Proximal:Active,Screen:Signif,0.4354863,0.4354863,Mean(ZScore),Mean(ZScore),CRISPRi_Growth_K562_Gersbach_JinWoo,CRISPRi-Growth,CRISPRi-Growth


**Check count**

In [9]:
lst = lst_dat_region_merge_crispri_arrange
lst = lapply(lst, function(dat){
    res = table(dat$Assay_Label, dat$Group, dnn = c("Assay", "Group"))
    dat = as.data.frame(res)
    return(dat)
})

dat = bind_rows(lst, .id = "Folder")
dat = dat %>% tidyr::spread(Folder, Freq)
fun_display_table(dat)

Assay,Group,fcc_astarr_macs_input_overlap,fcc_astarr_macs_input_union
CRISPRi-Growth,Distal:Active,369,394
CRISPRi-Growth,Proximal:Active,1059,1061


### Get the columns for TSS summary

In [10]:
lst = lst_dat_region_annot_tss_import
lst = lapply(lst, function(dat){
    dat = dat %>%
        dplyr::select(Chrom:Region, TSS_Proximity, TSS_Category) %>%
        dplyr::distinct()
    return(dat)
})

### assign and show
lst_dat_region_annot_tss_arrange = lst

res = lapply(lst, dim)
print(res)

dat = lst[[1]]
fun_display_table(head(dat, 3))

$fcc_astarr_macs_input_overlap
[1] 150042      6

$fcc_astarr_macs_input_union
[1] 246852      6



Chrom,ChromStart,ChromEnd,Region,TSS_Proximity,TSS_Category
chr1,10038,10405,chr1:10038-10405,Proximal,Proximal:Non-Essential
chr1,14282,14614,chr1:14282-14614,Distal,Distal
chr1,16025,16338,chr1:16025-16338,Proximal,Proximal:Non-Essential


## Join the tables

In [11]:
### loop to import data
lst = lapply(VEC_TXT_FOLDER, function(txt_folder){
    ### get tables
    dat_region_merge = lst_dat_region_merge_crispri_arrange[[txt_folder]]
    dat_region_annot = lst_dat_region_annot_tss_arrange[[txt_folder]]

    ### join tables
    dat = dplyr::left_join(
        dat_region_merge,
        dat_region_annot,
        by = c("Chrom", "ChromStart", "ChromEnd", "Region"))
    return(dat)
})

### assign and show
lst_dat_region_merge_crispri_tss = lst

res = lapply(lst, dim)
print(res)

dat = lst[[1]]
head(dat, 3)

$fcc_astarr_macs_input_overlap
[1] 1428   15

$fcc_astarr_macs_input_union
[1] 1455   15



Chrom,ChromStart,ChromEnd,Region,Group,Label,Score,Zcore,Score_Label,Zcore_Label,Assay_Folder,Assay_Group,Assay_Label,TSS_Proximity,TSS_Category
<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
chr1,1068587,1070616,chr1:1068587-1070616,Proximal:Active,Screen:Signif,0.2520571,0.2520571,Mean(ZScore),Mean(ZScore),CRISPRi_Growth_K562_Gersbach_JinWoo,CRISPRi-Growth,CRISPRi-Growth,Proximal,Proximal:Non-Essential
chr1,1324275,1325765,chr1:1324275-1325765,Proximal:Active,Screen:Signif,0.5069515,0.5069515,Mean(ZScore),Mean(ZScore),CRISPRi_Growth_K562_Gersbach_JinWoo,CRISPRi-Growth,CRISPRi-Growth,Proximal,Proximal:Non-Essential
chr1,1573998,1575422,chr1:1573998-1575422,Proximal:Active,Screen:Signif,0.4354863,0.4354863,Mean(ZScore),Mean(ZScore),CRISPRi_Growth_K562_Gersbach_JinWoo,CRISPRi-Growth,CRISPRi-Growth,Proximal,Proximal:Essential


## Export results

In [12]:
for (txt_folder in VEC_TXT_FOLDER){
    ### get tables
    dat_region_merge = lst_dat_region_merge_crispri_tss[[txt_folder]]
    
    ### set file directory
    txt_fdiry  = file.path(FD_RES, "analysis_fcc_comparison", txt_folder)
    txt_fname = "region.annotation.tss_essential.coverage.crispri_growth.tsv"
    txt_fpath = file.path(txt_fdiry, txt_fname)
    
    ### write table
    dat = dat_region_merge
    write_tsv(dat, txt_fpath)
}